In [1]:
import os
import glayout
from glayout import gf180, nmos, rename_ports_by_orientation
from glayout.backend import Component

gf180.activate()

def make_unit(dummies):
    return nmos(
        pdk=gf180,
        width=3.0,
        length=None,
        fingers=1,
        multipliers=1,
        with_tie=False,
        with_dummy=dummies,
        with_dnwell=False,
        with_substrate_tap=False,
        dummy_routes=True,
        sd_route_topmet="met2",
        gate_route_topmet="met2",
        sd_route_left=True,
        sd_rmult=1,
        gate_rmult=1,
        interfinger_rmult=1,
    )

unit_L = make_unit((True, False))
unit_R = make_unit((False, True))

def size(component):
    (xmin, ymin), (xmax, ymax) = component.bbox
    return float(xmax - xmin), float(ymax - ymin)

wL, hL = size(unit_L)
wR, hR = size(unit_R)

x_pitch = max(wL, wR) + 2.0
y_pitch = max(hL, hR) + 2.0

top = Component(name="nfet_2x2_diffpair")
device_refs = []
pattern = [["A0", "B0"], ["B1", "A1"]]

for row in range(2):
    for col in range(2):
        unit = unit_L if col == 0 else unit_R
        ref = top << unit

        # Official gLayout common-centroid orientation:
        # mirror the top row before translating it.
        if row == 0:
            ref = rename_ports_by_orientation(ref.mirror_y())

        x = (col - 0.5) * x_pitch
        y = (0.5 - row) * y_pitch

        ref.movex(x)
        ref.movey(y)
        device_refs.append(ref)

        print(f"{pattern[row][col]}: ({x:.3f}, {y:.3f})")

assert len(device_refs) == 4

def dump_ports(ref):
    ports = (
        ref.ports.items()
        if hasattr(ref.ports, "items")
        else [(p.name, p) for p in ref.get_ports_list()]
    )

    for name, port in sorted(ports):
        try:
            glayer = gf180.layer_to_glayer(port.layer)
        except Exception:
            glayer = "unknown"

        print(
            f"{name:45s} "
            f"center={str(tuple(map(float, port.center))):24s} "
            f"layer={str(port.layer):12s} "
            f"glayer={glayer}"
        )


dump_ports(device_refs[0])

gds_path = os.path.abspath("nfet_2x2_diffpair_no_routing.gds")
top.write_gds(gds_path)

print(f"\nCreated 4-device AB/BA array:\nA0  B0\nB1  A1")
print(f"GDS: {gds_path}")

2026-06-19 05:33:35.418 | INFO     | gdsfactory.pdk:activate:337 - 'gf180' PDK is now active


A0: (-3.540, 4.065)


/tmp/ipykernel_22579/3354450623.py:90: UserWarning: Unnamed cells, 2 in 'nfet_2x2_diffpair'
  top.write_gds(gds_path)
2026-06-19 05:33:42.870 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/foss/designs/notebooks/nfet_2x2_diffpair_no_routing.gds'


B0: (3.540, 4.065)
B1: (-3.540, -4.065)
A1: (3.540, -4.065)
drain_E                                       center=(-1.42, 1.3699999999999997) layer=(36, 0)      glayer=met2
drain_N                                       center=(-2.24, 1.1199999999999997) layer=(36, 0)      glayer=met2
drain_S                                       center=(-2.24, 1.6199999999999997) layer=(36, 0)      glayer=met2
drain_W                                       center=(-3.06, 1.3699999999999997) layer=(36, 0)      glayer=met2
gate_E                                        center=(-1.99, 6.76)            layer=(36, 0)      glayer=met2
gate_N                                        center=(-2.24, 6.51)            layer=(36, 0)      glayer=met2
gate_S                                        center=(-2.24, 7.01)            layer=(36, 0)      glayer=met2
gate_W                                        center=(-2.49, 6.76)            layer=(36, 0)      glayer=met2
multiplier_0_diff_E                           center=(-1

In [2]:
# Build a semantic map of the four devices and their terminal ports.

assert len(device_refs) == 4
devices = dict(zip(("A0", "B0", "B1", "A1"), device_refs))
port_map = {}

for device_name, ref in devices.items():
    ports = dict(ref.ports.items())
    port_map[device_name] = {}

    print(f"\n{device_name}")

    for terminal in ("source", "drain", "gate"):
        prefix = f"multiplier_0_{terminal}_"
        matches = {
            name.removeprefix(prefix): port
            for name, port in ports.items()
            if name.startswith(prefix)
        }

        if not matches:
            raise RuntimeError(
                f"{device_name} has no ports beginning with '{prefix}'"
            )

        port_map[device_name][terminal] = matches

        for direction, port in sorted(matches.items()):
            try:
                glayer = gf180.layer_to_glayer(port.layer)
            except Exception:
                glayer = "unknown"

            x, y = map(float, port.center)
            print(
                f"  {terminal:6s} {direction:2s} "
                f"center=({x:7.3f}, {y:7.3f}) "
                f"layer={str(port.layer):10s} "
                f"glayer={glayer}"
            )

print("\nExample access:")
print('port_map["A0"]["source"]["E"]')


A0
  source E  center=( -1.420,   2.170) layer=(36, 0)    glayer=met2
  source N  center=( -2.240,   1.920) layer=(36, 0)    glayer=met2
  source S  center=( -2.240,   2.420) layer=(36, 0)    glayer=met2
  source W  center=( -3.060,   2.170) layer=(36, 0)    glayer=met2
  drain  E  center=( -1.420,   1.370) layer=(36, 0)    glayer=met2
  drain  N  center=( -2.240,   1.120) layer=(36, 0)    glayer=met2
  drain  S  center=( -2.240,   1.620) layer=(36, 0)    glayer=met2
  drain  W  center=( -3.060,   1.370) layer=(36, 0)    glayer=met2
  gate   E  center=( -1.990,   6.760) layer=(36, 0)    glayer=met2
  gate   N  center=( -2.240,   6.510) layer=(36, 0)    glayer=met2
  gate   S  center=( -2.240,   7.010) layer=(36, 0)    glayer=met2
  gate   W  center=( -2.490,   6.760) layer=(36, 0)    glayer=met2

B0
  source E  center=(  3.060,   2.170) layer=(36, 0)    glayer=met2
  source N  center=(  2.240,   1.920) layer=(36, 0)    glayer=met2
  source S  center=(  2.240,   2.420) layer=(36, 0)   

In [3]:
from glayout import straight_route

top_source_route = top << straight_route(
    pdk=gf180,
    edge1=port_map["A0"]["source"]["E"],
    edge2=port_map["B0"]["source"]["W"],
)

top.write_gds("nfet_2x2_top_source_test.gds")
print("Top source pair connected.")

/tmp/ipykernel_22579/634076918.py:9: UserWarning: Unnamed cells, 2 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_top_source_test.gds")
2026-06-19 05:33:42.961 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_top_source_test.gds'


Top source pair connected.


In [4]:
bottom_source_route = top << straight_route(
    pdk=gf180,
    edge1=port_map["B1"]["source"]["E"],
    edge2=port_map["A1"]["source"]["W"],
)

top.write_gds("nfet_2x2_both_source_pairs.gds")
print("Top and bottom source pairs connected.")

/tmp/ipykernel_22579/35955455.py:7: UserWarning: Unnamed cells, 2 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_both_source_pairs.gds")
2026-06-19 05:33:43.008 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_both_source_pairs.gds'


Top and bottom source pairs connected.


In [5]:
import inspect
from glayout import via_stack

print(inspect.signature(via_stack))

m2_m3_via = via_stack(
    gf180,
    "met2",
    "met3",
    centered=True,
)

for name, port in sorted(m2_m3_via.ports.items()):
    print(
        f"{name:30s} "
        f"center={tuple(map(float, port.center))} "
        f"layer={port.layer}"
    )

(pdk: glayout.pdk.mappedpdk.MappedPDK, glayer1: str, glayer2: str, centered: bool = True, fullbottom: bool = False, fulltop: bool = False, assume_bottom_via: bool = False, same_layer_behavior: Literal['lay_nothing', 'min_square'] = 'lay_nothing') -> gdsfactory.component.Component
bottom_layer_E                 center=(0.25, 0.0) layer=(36, 0)
bottom_layer_N                 center=(0.0, 0.25) layer=(36, 0)
bottom_layer_S                 center=(0.0, -0.25) layer=(36, 0)
bottom_layer_W                 center=(-0.25, 0.0) layer=(36, 0)
bottom_met_E                   center=(0.25, 0.0) layer=(36, 0)
bottom_met_N                   center=(0.0, 0.25) layer=(36, 0)
bottom_met_S                   center=(0.0, -0.25) layer=(36, 0)
bottom_met_W                   center=(-0.25, 0.0) layer=(36, 0)
bottom_via_E                   center=(0.13, 0.0) layer=(38, 0)
bottom_via_N                   center=(0.0, 0.13) layer=(38, 0)
bottom_via_S                   center=(0.0, -0.13) layer=(38, 0)
bottom_via

In [6]:
from glayout import via_stack, straight_route

via = via_stack(
    pdk=gf180,
    glayer1="met2",
    glayer2="met3",
    centered=True,
)

# Centers of the two existing M2 source straps
x_top = (
    float(port_map["A0"]["source"]["E"].center[0])
    + float(port_map["B0"]["source"]["W"].center[0])
) / 2
y_top = float(port_map["A0"]["source"]["E"].center[1])

x_bot = (
    float(port_map["B1"]["source"]["E"].center[0])
    + float(port_map["A1"]["source"]["W"].center[0])
) / 2
y_bot = float(port_map["B1"]["source"]["E"].center[1])

via_top = top << via
via_top.movex(x_top)
via_top.movey(y_top)

via_bot = top << via
via_bot.movex(x_bot)
via_bot.movey(y_bot)

source_vertical = top << straight_route(
    pdk=gf180,
    edge1=via_top.ports["top_met_S"],
    edge2=via_bot.ports["top_met_N"],
)

top.write_gds("nfet_2x2_common_source.gds")
print("Common-source net joined on M3.")

/tmp/ipykernel_22579/523336400.py:37: UserWarning: Unnamed cells, 2 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_common_source.gds")
2026-06-19 05:33:43.134 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_common_source.gds'


Common-source net joined on M3.


In [7]:
from glayout import via_stack
from glayout.backend import rectangle

via23 = via_stack(pdk=gf180, glayer1="met2", glayer2="met3")
via24 = via_stack(pdk=gf180, glayer1="met2", glayer2="met4")

def place_via(cell, port):
    ref = top << cell
    ref.movex(float(port.center[0]))
    ref.movey(float(port.center[1]))
    return ref

def add_path(points, glayer, width):
    layer = gf180.get_glayer(glayer)

    for (x1, y1), (x2, y2) in zip(points, points[1:]):
        if x1 != x2 and y1 != y2:
            raise ValueError("Only Manhattan segments are allowed.")

        size = (
            (abs(x2 - x1) + width, width)
            if y1 == y2
            else (width, abs(y2 - y1) + width)
        )

        ref = top << rectangle(size=size, layer=layer, centered=True)
        ref.movex((x1 + x2) / 2)
        ref.movey((y1 + y2) / 2)

snap = lambda x: float(gf180.snap_to_2xgrid(x))

w3 = snap(gf180.get_grule("met3")["min_width"])
w4 = snap(gf180.get_grule("met4")["min_width"])
s3 = snap(gf180.get_grule("met3")["min_separation"])
s4 = snap(gf180.get_grule("met4")["min_separation"])

clearance = max(w3 + s3, w4 + s4)
outer_x = snap(
    max(abs(float(top.xmin)), abs(float(top.xmax)))
    + clearance
    + max(float(via23.xsize), float(via24.xsize)) / 2
)

source_top_y = float(port_map["A0"]["source"]["E"].center[1])
source_bot_y = float(port_map["B1"]["source"]["E"].center[1])
via_half = max(float(via23.ysize), float(via24.ysize)) / 2

y_upper = snap(source_top_y + via_half + clearance)
y_lower = snap(source_bot_y - via_half - clearance)

# Drain A: A0 ↔ A1 on M3
a0 = port_map["A0"]["drain"]["W"]
a1 = port_map["A1"]["drain"]["E"]

place_via(via23, a0)
place_via(via23, a1)

ax0, ay0 = map(float, a0.center)
ax1, ay1 = map(float, a1.center)

add_path(
    [
        (ax0, ay0),
        (-outer_x, ay0),
        (-outer_x, y_lower),
        (ax1, y_lower),
        (ax1, ay1),
    ],
    "met3",
    w3,
)

# Drain B: B0 ↔ B1 on M4
b0 = port_map["B0"]["drain"]["E"]
b1 = port_map["B1"]["drain"]["W"]

place_via(via24, b0)
place_via(via24, b1)

bx0, by0 = map(float, b0.center)
bx1, by1 = map(float, b1.center)

add_path(
    [
        (bx0, by0),
        (outer_x, by0),
        (outer_x, y_upper),
        (bx1, y_upper),
        (bx1, by1),
    ],
    "met4",
    w4,
)

top.write_gds("nfet_2x2_corrected_drains.gds")
print("Drain A: A0 ↔ A1 on M3")
print("Drain B: B0 ↔ B1 on M4")

/tmp/ipykernel_22579/1288401207.py:95: UserWarning: Unnamed cells, 2 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_corrected_drains.gds")
2026-06-19 05:33:43.242 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_corrected_drains.gds'


Drain A: A0 ↔ A1 on M3
Drain B: B0 ↔ B1 on M4


In [8]:
from glayout import via_stack

via23_gate = via_stack(
    pdk=gf180,
    glayer1="met2",
    glayer2="met3",
)

via24_gate = via_stack(
    pdk=gf180,
    glayer1="met2",
    glayer2="met4",
)

w3 = snap(gf180.get_grule("met3")["min_width"])
w4 = snap(gf180.get_grule("met4")["min_width"])
s3 = snap(gf180.get_grule("met3")["min_separation"])
s4 = snap(gf180.get_grule("met4")["min_separation"])

clearance = max(w3 + s3, w4 + s4)
via_half = max(
    float(via23_gate.ysize),
    float(via24_gate.ysize),
) / 2

gate_x = snap(
    max(abs(float(top.xmin)), abs(float(top.xmax)))
    + clearance
    + via_half
)

gate_y_top = snap(float(top.ymax) + clearance + via_half)
gate_y_bot = snap(float(top.ymin) - clearance - via_half)

# Gate A: A0 ↔ A1 on M3
a0g = port_map["A0"]["gate"]["W"]
a1g = port_map["A1"]["gate"]["E"]

place_via(via23_gate, a0g)
place_via(via23_gate, a1g)

a0x, a0y = map(float, a0g.center)
a1x, a1y = map(float, a1g.center)

add_path(
    [
        (a0x, a0y),
        (a0x, gate_y_top),
        (-gate_x, gate_y_top),
        (-gate_x, gate_y_bot),
        (a1x, gate_y_bot),
        (a1x, a1y),
    ],
    "met3",
    w3,
)

# Gate B: B0 ↔ B1 on M4
b0g = port_map["B0"]["gate"]["E"]
b1g = port_map["B1"]["gate"]["W"]

place_via(via24_gate, b0g)
place_via(via24_gate, b1g)

b0x, b0y = map(float, b0g.center)
b1x, b1y = map(float, b1g.center)

add_path(
    [
        (b0x, b0y),
        (b0x, gate_y_top),
        (gate_x, gate_y_top),
        (gate_x, gate_y_bot),
        (b1x, gate_y_bot),
        (b1x, b1y),
    ],
    "met4",
    w4,
)

top.write_gds("nfet_2x2_corrected_gate_routing.gds")

/tmp/ipykernel_22579/183261037.py:81: UserWarning: Unnamed cells, 2 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_corrected_gate_routing.gds")
2026-06-19 05:33:43.348 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_corrected_gate_routing.gds'


PosixPath('nfet_2x2_corrected_gate_routing.gds')

In [9]:
from glayout import tapring

(xmin, ymin), (xmax, ymax) = top.bbox
xmin, ymin, xmax, ymax = map(float, (xmin, ymin, xmax, ymax))

margin = float(gf180.snap_to_2xgrid(
    max(
        gf180.util_max_metal_seperation(),
        gf180.get_grule("active_diff", "active_tap")["min_separation"],
    )
    + gf180.get_grule("p+s/d", "active_tap")["min_enclosure"]
))

bulk_ring = tapring(
    pdk=gf180,
    enclosed_rectangle=(
        xmax - xmin + 2 * margin,
        ymax - ymin + 2 * margin,
    ),
    sdlayer="p+s/d",
    horizontal_glayer="met2",
    vertical_glayer="met1",
)

bulk_ref = top << bulk_ring
bulk_ref.movex((xmin + xmax) / 2)
bulk_ref.movey((ymin + ymax) / 2)

top.add_ports(bulk_ref.get_ports_list(), prefix="bulk_")

top.write_gds("nfet_2x2_with_bulk_ring.gds")
print("Array-level p+ substrate ring added.")

/tmp/ipykernel_22579/1979890818.py:31: UserWarning: Unnamed cells, 3 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_with_bulk_ring.gds")
2026-06-19 05:33:44.153 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_with_bulk_ring.gds'


Array-level p+ substrate ring added.


In [10]:
devices = dict(zip(("A0", "B0", "B1", "A1"), device_refs))

print("\nBulk-ring ports:")
for name in sorted(bulk_ref.ports.keys()):
    if "top_met" in name:
        print(name)

print("\nDummy connection ports:")
for device_name, ref in devices.items():
    names = [
        name for name in ref.ports.keys()
        if "dummy" in name and "gsdcon_top_met" in name
    ]
    print(device_name, names)


Bulk-ring ports:
E_array_row0_col0_top_met_E
E_array_row0_col0_top_met_N
E_array_row0_col0_top_met_S
E_array_row0_col0_top_met_W
E_array_row10_col0_top_met_E
E_array_row10_col0_top_met_N
E_array_row10_col0_top_met_S
E_array_row10_col0_top_met_W
E_array_row11_col0_top_met_E
E_array_row11_col0_top_met_N
E_array_row11_col0_top_met_S
E_array_row11_col0_top_met_W
E_array_row12_col0_top_met_E
E_array_row12_col0_top_met_N
E_array_row12_col0_top_met_S
E_array_row12_col0_top_met_W
E_array_row13_col0_top_met_E
E_array_row13_col0_top_met_N
E_array_row13_col0_top_met_S
E_array_row13_col0_top_met_W
E_array_row14_col0_top_met_E
E_array_row14_col0_top_met_N
E_array_row14_col0_top_met_S
E_array_row14_col0_top_met_W
E_array_row15_col0_top_met_E
E_array_row15_col0_top_met_N
E_array_row15_col0_top_met_S
E_array_row15_col0_top_met_W
E_array_row16_col0_top_met_E
E_array_row16_col0_top_met_N
E_array_row16_col0_top_met_S
E_array_row16_col0_top_met_W
E_array_row17_col0_top_met_E
E_array_row17_col0_top_met_N


In [11]:
from glayout import straight_route

devices = {
    "A0": device_refs[0],
    "B0": device_refs[1],
    "B1": device_refs[2],
    "A1": device_refs[3],
}

dummy_connections = {
    "A0": ("multiplier_0_dummy_L_gsdcon_top_met_W", "W_top_met_W"),
    "B1": ("multiplier_0_dummy_L_gsdcon_top_met_W", "W_top_met_W"),
    "B0": ("multiplier_0_dummy_R_gsdcon_top_met_W", "E_top_met_E"),
    "A1": ("multiplier_0_dummy_R_gsdcon_top_met_W", "E_top_met_E"),
}

for name, (dummy_port, ring_port) in dummy_connections.items():
    assert dummy_port in devices[name].ports
    assert ring_port in bulk_ref.ports

    top << straight_route(
        pdk=gf180,
        edge1=devices[name].ports[dummy_port],
        edge2=bulk_ref.ports[ring_port],
        glayer2="met1",
    )

    print(f"{name} dummy → bulk ring")

top.write_gds("nfet_2x2_bulk_and_dummies.gds")

A0 dummy → bulk ring
B1 dummy → bulk ring
B0 dummy → bulk ring


/tmp/ipykernel_22579/2221602483.py:30: UserWarning: Unnamed cells, 3 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_bulk_and_dummies.gds")
2026-06-19 05:33:44.439 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_bulk_and_dummies.gds'


A1 dummy → bulk ring


PosixPath('nfet_2x2_bulk_and_dummies.gds')

In [12]:
def add_named_port(name, port, layer=None):
    top.add_port(
        name=name,
        center=tuple(map(float, port.center)),
        width=float(port.width),
        orientation=port.orientation,
        layer=layer or port.layer,
    )

# neutral electrical interfaces
add_named_port("GATE_A",  port_map["A0"]["gate"]["W"])
add_named_port("GATE_B",  port_map["B0"]["gate"]["E"])
add_named_port("DRAIN_A", port_map["A0"]["drain"]["W"])
add_named_port("DRAIN_B", port_map["B0"]["drain"]["E"])
add_named_port("TAIL",    via_top.ports["top_met_N"])      # common source on M3
add_named_port("BULK",    bulk_ref.ports["W_top_met_W"])   # tap ring

print("Top-level ports:")
for name in sorted(top.ports.keys()):
    if name in {"GATE_A", "GATE_B", "DRAIN_A", "DRAIN_B", "TAIL", "BULK"}:
        p = top.ports[name]
        print(name, tuple(map(float, p.center)), p.layer)

top.write_gds("nfet_2x2_named_ports.gds")

/tmp/ipykernel_22579/699402651.py:24: UserWarning: Unnamed cells, 3 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_named_ports.gds")
2026-06-19 05:33:44.521 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_named_ports.gds'


Top-level ports:
BULK (-8.89, 0.0) (34, 0)
DRAIN_A (-3.06, 1.37) (36, 0)
DRAIN_B (3.06, 1.37) (36, 0)
GATE_A (-2.49, 6.76) (36, 0)
GATE_B (2.49, 6.76) (36, 0)
TAIL (0.0, 2.42) (42, 0)


PosixPath('nfet_2x2_named_ports.gds')

In [13]:
import os
import shutil
import subprocess

klayout_path = "/foss/tools/klayout/klayout"  # replace with actual result
klayout_dir = os.path.dirname(klayout_path)

os.environ["PATH"] = klayout_dir + ":" + os.environ["PATH"]

print("KLayout now found at:", shutil.which("klayout"))

subprocess.run(["klayout", "-v"], check=True)

report = gf180.drc(
    top,
    "gf180_nfet_2x2_named_ports.lyrdb",
)
print(report)

KLayout now found at: /foss/tools/klayout/klayout


/foss/designs/gLayout/src/glayout/pdk/mappedpdk.py:333: UserWarning: Unnamed cells, 3 in 'nfet_2x2_diffpair'
  layout_path = Path(layout.write_gds(gdsdir=tempdir.name)).resolve()
2026-06-19 05:33:44.868 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to '/tmp/tmpiy10ja95/nfet_2x2_diffpair.gds'


KLayout 0.30.2
KLayout version: 0.30.2
2026-06-19 05:33:47 +0200: Memory Usage (660972K) : Starting running GF180MCU Klayout DRC runset on /tmp/tmpiy10ja95/nfet_2x2_diffpair.gds
2026-06-19 05:33:47 +0200: Memory Usage (660972K) : Loading database to memory is complete.
2026-06-19 05:33:47 +0200: Memory Usage (661036K) : GF180MCU Klayout DRC runset output at: /foss/designs/notebooks/gf180_nfet_2x2_named_ports.lyrdb/gf180_nfet_2x2_diffpair_drcreport.lyrdb
2026-06-19 05:33:47 +0200: Memory Usage (661036K) : Number of threads to use 4
2026-06-19 05:33:47 +0200: Memory Usage (661036K) : flat  mode is enabled.
2026-06-19 05:33:47 +0200: Memory Usage (661036K) : Read in polygons from layers.
2026-06-19 05:33:47 +0200: Memory Usage (662060K) : Starting deriving base layers.
2026-06-19 05:33:47 +0200: Memory Usage (662496K) : Evaluate switches.
2026-06-19 05:33:47 +0200: Memory Usage (662496K) : FEOL is enabled.
2026-06-19 05:33:47 +0200: Memory Usage (662496K) : BEOL is enabled.
2026-06-19 05:

In [14]:
def add_net_label(name, port):
    glayer = gf180.layer_to_glayer(port.layer)

    try:
        label_layer = gf180.get_glayer(f"{glayer}_label")
    except Exception:
        label_layer = port.layer

    top.add_label(
        text=name,
        position=tuple(map(float, port.center)),
        layer=label_layer,
    )

labels = {
    "GATE_A":  port_map["A0"]["gate"]["W"],
    "GATE_B":  port_map["B0"]["gate"]["E"],
    "DRAIN_A": port_map["A0"]["drain"]["W"],
    "DRAIN_B": port_map["B0"]["drain"]["E"],
    "TAIL":    via_top.ports["top_met_N"],
    "BULK":    bulk_ref.ports["W_top_met_W"],
}

for name, port in labels.items():
    add_net_label(name, port)
    print(f"labeled {name} on {gf180.layer_to_glayer(port.layer)}")

top.write_gds("nfet_2x2_labeled_for_lvs.gds")

/tmp/ipykernel_22579/888553905.py:28: UserWarning: Unnamed cells, 3 in 'nfet_2x2_diffpair'
  top.write_gds("nfet_2x2_labeled_for_lvs.gds")
2026-06-19 05:33:56.074 | INFO     | gdsfactory.component:_write_library:1851 - Wrote to 'nfet_2x2_labeled_for_lvs.gds'


labeled GATE_A on met2
labeled GATE_B on met2
labeled DRAIN_A on met2
labeled DRAIN_B on met2
labeled TAIL on met3
labeled BULK on met1


PosixPath('nfet_2x2_labeled_for_lvs.gds')

In [15]:
netgen_path = "/foss/tools/netgen/bin/netgen"  # replace with actual result
os.environ["PATH"] = os.path.dirname(netgen_path) + ":" + os.environ["PATH"]

magic_path = "/foss/tools/magic/bin/magic"  # replace with actual result
os.environ["PATH"] = os.path.dirname(magic_path) + ":" + os.environ["PATH"]

print("netgen:", shutil.which("netgen"))

from pathlib import Path

Path("probe_ref.spice").write_text("""
.subckt nfet_2x2_diffpair GATE_A GATE_B DRAIN_A DRAIN_B TAIL BULK
.ends nfet_2x2_diffpair
""")

try:
    result = gf180.lvs_netgen(
        layout="nfet_2x2_labeled_for_lvs.gds",
        design_name="nfet_2x2_diffpair",
        netlist="probe_ref.spice",
        output_file_path="nfet_2x2_probe_lvs.txt",
        copy_intermediate_files=True,
        show_scripts=True,
    )
    print(result)
except Exception as e:
    print("Probe LVS ended with:")
    print(type(e).__name__, e)

print("\nLikely generated LVS/extraction files:")
for p in Path(".").rglob("*"):
    name = p.name.lower()
    if any(k in name for k in ["lvs", "ext", "spice", "cdl", "netgen"]):
        if p.is_file():
            print(p)

netgen: /foss/tools/netgen/bin/netgen
using user specified pdk_root, will search for required files in the specified directory
Design name from CDL file: nfet_2x2_diffpair matches the design name: nfet_2x2_diffpair
Creating magic script for LVS...
==== MAGIC SCRIPT BEGIN ====
drc off
gds flatglob *\$\$*
gds read /tmp/tmpn_7jhrzh/nfet_2x2_diffpair.gds

# LVS Netlist — extract transistors only. Skipping ext2resist/extresist on
# the lvsmag output keeps long routes from showing up as thousands of
# parasitic resistors that the schematic does not model (e.g. opamp's
# 7000+ r:N instances would otherwise swamp the comparison).
load nfet_2x2_diffpair
select top cell

extract all

ext2spice lvs
# Force the top cell to be wrapped in a `.subckt {design_name}` block.
# Without this, magic emits the top circuit as flat top-level cards, and
# netgen reports `Cannot find cell {design_name}` when given the file —
# which silently aborts before any report is written. See diff_to_single
# / transmissi

In [16]:
from pathlib import Path

for p in Path(".").rglob("*"):
    name = p.name.lower()
    if p.is_file() and any(k in name for k in ["ext", "layout", "spice", "cdl"]):
        if p.suffix.lower() in [".spice", ".sp", ".cdl", ".net", ".cir"]:
            print("\n" + "=" * 80)
            print(p)
            print("=" * 80)
            print(p.read_text(errors="ignore")[:4000])


nfet_2x2_ref_flat.spice

* Reference schematic for 2x2 AB/BA NFET differential-pair input core

.subckt nfet_2x2_diffpair GATE_A GATE_B DRAIN_A DRAIN_B TAIL BULK

XA0 DRAIN_A GATE_A TAIL BULK nfet_03v3 w=3u l=0.28u
XA1 DRAIN_A GATE_A TAIL BULK nfet_03v3 w=3u l=0.28u
XB0 DRAIN_B GATE_B TAIL BULK nfet_03v3 w=3u l=0.28u
XB1 DRAIN_B GATE_B TAIL BULK nfet_03v3 w=3u l=0.28u

XDA0 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u
XDA1 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u
XDB0 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u
XDB1 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u

.ends nfet_2x2_diffpair


probe_ref.spice

.subckt nfet_2x2_diffpair GATE_A GATE_B DRAIN_A DRAIN_B TAIL BULK
.ends nfet_2x2_diffpair


nfet_2x2_lvs_result.txt/lvs/nfet_2x2_diffpair/nfet_2x2_diffpair_lvsmag.spice
* NGSPICE file created from nfet_2x2_diffpair.ext - technology: gf180mcuD

.subckt Unnamed_b5c454ed a_288_n381# a_n438_n381# a_82_n381# a_210_n589# VSUBS
X0 a_288_n381# a_210_n589# a_82_n381# VSUBS nfet_03v3 ad=2.25p pd=7.

In [17]:
from pathlib import Path

Path("nfet_2x2_ref_flat.spice").write_text("""
* Reference schematic for 2x2 AB/BA NFET differential-pair input core

.subckt nfet_2x2_diffpair GATE_A GATE_B DRAIN_A DRAIN_B TAIL BULK

* Active matched devices
XA0 DRAIN_A GATE_A TAIL BULK nfet_03v3 w=3u l=0.28u
XA1 DRAIN_A GATE_A TAIL BULK nfet_03v3 w=3u l=0.28u
XB0 DRAIN_B GATE_B TAIL BULK nfet_03v3 w=3u l=0.28u
XB1 DRAIN_B GATE_B TAIL BULK nfet_03v3 w=3u l=0.28u

* Perimeter dummy devices tied to bulk
XDA0 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u
XDA1 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u
XDB0 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u
XDB1 BULK BULK BULK BULK nfet_03v3 w=3u l=0.28u

.ends nfet_2x2_diffpair
""")

lvs_result = gf180.lvs_netgen(
    layout="nfet_2x2_labeled_for_lvs.gds",
    design_name="nfet_2x2_diffpair",
    netlist="nfet_2x2_ref_flat.spice",
    output_file_path="nfet_2x2_lvs_result.txt",
    copy_intermediate_files=True,
    show_scripts=True,
)

print(lvs_result)

using user specified pdk_root, will search for required files in the specified directory
Design name from CDL file: nfet_2x2_diffpair matches the design name: nfet_2x2_diffpair
Creating magic script for LVS...
==== MAGIC SCRIPT BEGIN ====
drc off
gds flatglob *\$\$*
gds read /tmp/tmpds12shxc/nfet_2x2_diffpair.gds

# LVS Netlist — extract transistors only. Skipping ext2resist/extresist on
# the lvsmag output keeps long routes from showing up as thousands of
# parasitic resistors that the schematic does not model (e.g. opamp's
# 7000+ r:N instances would otherwise swamp the comparison).
load nfet_2x2_diffpair
select top cell

extract all

ext2spice lvs
# Force the top cell to be wrapped in a `.subckt {design_name}` block.
# Without this, magic emits the top circuit as flat top-level cards, and
# netgen reports `Cannot find cell {design_name}` when given the file —
# which silently aborts before any report is written. See diff_to_single
# / transmission_gate ERRORs.
ext2spice subcircuit t